In [2]:
import seaborn as sns
df = sns.load_dataset('tips')
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [4]:
df['day'].unique()

['Sun', 'Sat', 'Thur', 'Fri']
Categories (4, object): ['Thur', 'Fri', 'Sat', 'Sun']

In [5]:
df['time'].unique()

['Dinner', 'Lunch']
Categories (2, object): ['Lunch', 'Dinner']

In [6]:
df.isnull().sum()

,0
total_bill,0
tip,0
sex,0
smoker,0
day,0
time,0
size,0


In [7]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df['time'] = encoder.fit_transform(df['time'])

In [9]:
df['time'].unique()

array([0, 1])

In [13]:
x = df.drop('time', axis=1)
y = df['time']

In [14]:
x.head()

,total_bill,tip,sex,smoker,day,size
0,16.99,1.01,Female,No,Sun,2
1,10.34,1.66,Male,No,Sun,3
2,21.01,3.50,Male,No,Sun,3
3,23.68,3.31,Male,No,Sun,2
4,24.59,3.61,Female,No,Sun,4


In [15]:
y

,time
0,0
1,0
2,0
3,0
4,0
...,...
239,0
240,0
241,0
242,0


In [16]:
x['day'].value_counts()

,count
day,
Sat,87
Sun,76
Thur,62
Fri,19


In [19]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state = 42)

In [30]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [21]:
cat_cols=['sex','smoker', 'day']
num_cols=['total_bill', 'tip', 'size']

In [28]:

# Numerical pipeline
num_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

# Categorical pipeline
cat_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('OneHotEncoder', OneHotEncoder()

        )
    ]
)


In [31]:
prosessor = ColumnTransformer([
    ('num_pipeline',num_pipeline,num_cols),
    ('cat_pipeline',cat_pipeline,cat_cols)

])

In [33]:
x_train=prosessor.fit_transform(x_train)
x_test=prosessor.transform(x_test)


In [44]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

models = {'RandomForest':RandomForestClassifier()
          ,'LogisticRegression':LogisticRegression()
          ,'DecisionTree':DecisionTreeClassifier()

          }


In [41]:
def evaluate_model(x_train,x_test,y_train,y_test,models):
    report = {}
    for i in range(len(models)):
        model = list(models.values())[i]
        model.fit(x_train,y_train)
        y_test_pred = model.predict(x_test)
        test_model_score = accuracy_score(y_test,y_test_pred)
        report[list(models.keys())[i]]=test_model_score
    return report

In [46]:
evaluate_model(x_train, x_test, y_train, y_test, models)

{'RandomForest': 0.9591836734693877,
 'LogisticRegression': 1.0,
 'DecisionTree': 0.9387755102040817}

In [48]:
clasifier = RandomForestClassifier()


In [49]:
#hyperparameter tuning
params={'max_depth':[3,5,10,None],
        'n_estimators':[10,20,50,100],
        'criterion':['gini','entropy']
        }




In [52]:
from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=clasifier,param_distributions=params,n_iter=10,cv=5,verbose=3,n_jobs=-1,scoring='accuracy')
random_search.fit(x_train,y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(), n_jobs=-1,
                   param_distributions={'criterion': ['gini', 'entropy'],
                                        'max_depth': [3, 5, 10, None],
                                        'n_estimators': [10, 20, 50, 100]},
                   scoring='accuracy', verbose=3)

In [53]:
cv_best_params = random_search.best_params_
cv_best_params

{'n_estimators': 50, 'max_depth': None, 'criterion': 'entropy'}